# तन्त्र (Tantra-LLM) — Kaggle GPU Training
**अतुल्य AI | Atulya AI**

This notebook:
1. Clones the Tantra-LLM code from GitHub
2. Copies datasets from Kaggle Dataset (`atultantra/tantrads`)
3. Copies/resumes from checkpoint if available (`atultantra/tantra-llm`)
4. Trains on GPU with best settings
5. Saves checkpoints back to `/kaggle/working/` for download

> **Requirements**: Enable GPU (T4 x2) + Internet in notebook settings.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Verify GPU & Environment
# ─────────────────────────────────────────────────────────────
import os, subprocess

# Check GPU
print('=== GPU INFO ===')
os.system('nvidia-smi')

# Check disk space
print('\n=== DISK SPACE ===')
os.system('df -h /kaggle/working')

# Check available RAM
import psutil
ram = psutil.virtual_memory()
print(f'\n=== RAM: {ram.available/1024**3:.1f} GB available / {ram.total/1024**3:.1f} GB total ===')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Install Dependencies
# ─────────────────────────────────────────────────────────────
print('Installing dependencies...')
os.system('pip install -q tokenizers>=0.19.0 sentencepiece zstandard psutil py-cpuinfo rich '
          'dataclasses-json einops pyarrow scikit-learn edge-tts faster-whisper '
          'fastapi uvicorn jinja2 python-multipart morfessor brotli')
print('✅ Dependencies installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Clone Tantra-LLM from GitHub
# ─────────────────────────────────────────────────────────────
WORK_DIR = '/kaggle/working/Tantra-LLM'

if not os.path.exists(WORK_DIR):
    print('Cloning Tantra-LLM from GitHub...')
    os.system('git clone https://github.com/atulyaai/Tantra-LLM.git /kaggle/working/Tantra-LLM')
    print('✅ Cloned successfully.')
else:
    print('Repo already exists — pulling latest...')
    os.system('cd /kaggle/working/Tantra-LLM && git pull origin main')
    print('✅ Up to date.')

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', os.listdir('.'))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Copy Datasets from Kaggle Dataset
# Dataset: https://www.kaggle.com/datasets/atultantra/tantrads
# ─────────────────────────────────────────────────────────────
import shutil

DATASET_INPUT = '/kaggle/input/tantrads'
DATASETS_DIR  = f'{WORK_DIR}/Datasets'
os.makedirs(DATASETS_DIR, exist_ok=True)

if os.path.exists(DATASET_INPUT):
    files = os.listdir(DATASET_INPUT)
    print(f'Found {len(files)} files in dataset input:')
    for f in sorted(files):
        src = os.path.join(DATASET_INPUT, f)
        dst = os.path.join(DATASETS_DIR, f)
        size_mb = os.path.getsize(src) / (1024*1024)
        if not os.path.exists(dst):
            print(f'  Copying {f} ({size_mb:.1f} MB)...')
            shutil.copy2(src, dst)
        else:
            print(f'  ✓ {f} already exists ({size_mb:.1f} MB), skipping.')
    print('\n✅ Dataset files ready.')
else:
    print('⚠️  Kaggle dataset not found at', DATASET_INPUT)
    print('   → Add dataset via: Notebook Settings → Add Input → Dataset → atultantra/tantrads')
    # Fallback: try to generate minimal gold datasets
    print('   → Generating minimal fallback gold corpus for training...')
    os.system(f'cd {WORK_DIR} && python -c "from Tantra.dataset import generate_gold_datasets; generate_gold_datasets(force=True)"')

# Verify critical files
for required in ['master_train.jsonl', 'master_val.jsonl']:
    p = os.path.join(DATASETS_DIR, required)
    if os.path.exists(p):
        print(f'  ✅ {required}: {os.path.getsize(p)/(1024*1024):.1f} MB')
    else:
        print(f'  ❌ MISSING: {required}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Copy Checkpoint from Kaggle Model (Resume if available)
# Model: https://www.kaggle.com/models/atultantra/tantra-llm
# ─────────────────────────────────────────────────────────────
MODEL_INPUT   = '/kaggle/input/tantra-llm'  # Kaggle model path
MODEL_DIR     = f'{WORK_DIR}/Model'
BEST_DIR      = f'{WORK_DIR}/Model/Best'
LATEST_DIR    = f'{WORK_DIR}/Model/Latest'
CKPT_DIR      = f'{WORK_DIR}/Model/Checkpoints'

for d in [MODEL_DIR, BEST_DIR, LATEST_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

RESUME = False

if os.path.exists(MODEL_INPUT):
    model_files = []
    for root, dirs, files in os.walk(MODEL_INPUT):
        for f in files:
            model_files.append(os.path.join(root, f))

    print(f'Found {len(model_files)} files in Kaggle model input:')
    for src in sorted(model_files):
        fname = os.path.basename(src)
        size_mb = os.path.getsize(src) / (1024*1024)
        # Copy .pt checkpoint to Best and Latest for resuming
        if fname.endswith('.pt'):
            for dst_dir in [BEST_DIR, LATEST_DIR]:
                dst = os.path.join(dst_dir, 'checkpoint_latest.pt')
                if not os.path.exists(dst):
                    print(f'  Copying checkpoint {fname} ({size_mb:.1f} MB) → {dst_dir}...')
                    shutil.copy2(src, dst)
            RESUME = True
        elif fname in ['tokenizer.json', 'vocab.json', 'merges.txt',
                       'special_tokens_map.json', 'tokenizer_config.json']:
            dst = os.path.join(MODEL_DIR, fname)
            if not os.path.exists(dst):
                print(f'  Copying {fname} ({size_mb:.1f} MB)...')
                shutil.copy2(src, dst)

    if RESUME:
        print('\n✅ Checkpoint found → will RESUME training.')
    else:
        print('\n✅ Tokenizer files copied → will start FRESH training.')
else:
    print('ℹ️  No Kaggle model checkpoint found — starting fresh training.')
    print('   (Tokenizer will be loaded from repo Model/ directory)')

# Confirm tokenizer exists
tok_path = os.path.join(MODEL_DIR, 'tokenizer.json')
if os.path.exists(tok_path):
    print(f'\n✅ Tokenizer ready: {tok_path} ({os.path.getsize(tok_path)/1024**2:.1f} MB)')
else:
    print('\n❌ tokenizer.json missing — it should be in the repo (Model/tokenizer.json)')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Training Configuration
# ─────────────────────────────────────────────────────────────
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_COUNT = torch.cuda.device_count() if DEVICE == 'cuda' else 0

# Tune batch size & seq_len based on GPU VRAM
if GPU_COUNT >= 2:
    BATCH_SIZE = 16
    GRAD_ACCUM = 2
    SEQ_LEN    = 256
    print(f'✅ Dual GPU detected ({GPU_COUNT}x T4) — max batch settings')
elif GPU_COUNT == 1:
    BATCH_SIZE = 8
    GRAD_ACCUM = 4
    SEQ_LEN    = 256
    print(f'✅ Single GPU detected (T4) — moderate batch settings')
else:
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    SEQ_LEN    = 128
    print(f'⚠️  CPU mode — slow but functional')

STEPS = 5000
EVAL_EVERY = 200
CHECKPOINT_EVERY = 500
LR = '3e-4'
DIM = 512
LAYERS = 8
HEADS = 8

TRAIN_FILE = 'Datasets/master_train.jsonl'
VAL_FILE   = 'Datasets/master_val.jsonl'

print(f'\n=== TRAINING CONFIG ===')
print(f'Device    : {DEVICE} ({GPU_COUNT} GPUs)')
print(f'Batch     : {BATCH_SIZE} x grad_accum {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective batch')
print(f'Seq Len   : {SEQ_LEN}')
print(f'Steps     : {STEPS}')
print(f'Resume    : {RESUME}')
print(f'Train file: {TRAIN_FILE}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7: START TRAINING
# ─────────────────────────────────────────────────────────────
import subprocess, sys

# Build the command
if RESUME:
    CMD = [
        sys.executable, 'main.py',
        '--mode', 'auto-pilot',
        '--resume',
        '--dataset', TRAIN_FILE,
        '--val-dataset', VAL_FILE,
        '--steps', str(STEPS),
        '--device', DEVICE,
        '--batch-size', str(BATCH_SIZE),
        '--grad-accum', str(GRAD_ACCUM),
        '--seq-len', str(SEQ_LEN),
        '--eval-every', str(EVAL_EVERY),
        '--checkpoint-every', str(CHECKPOINT_EVERY),
    ]
    print('🔄 Resuming training from checkpoint...')
else:
    CMD = [
        sys.executable, 'main.py',
        '--mode', 'auto-pilot',
        '--dataset', TRAIN_FILE,
        '--val-dataset', VAL_FILE,
        '--steps', str(STEPS),
        '--auto-config',
        '--lr', LR,
        '--optimizer', 'lion',
        '--grad-accum', str(GRAD_ACCUM),
        '--batch-size', str(BATCH_SIZE),
        '--seq-len', str(SEQ_LEN),
        '--eval-every', str(EVAL_EVERY),
        '--checkpoint-every', str(CHECKPOINT_EVERY),
        '--device', DEVICE,
        '--dim', str(DIM),
        '--layers', str(LAYERS),
        '--heads', str(HEADS),
        '--training-stage', 'sft',
        '--mask-non-assistant',
    ]
    print('🚀 Starting fresh training...')

print('Command:', ' '.join(CMD))
print('='*70)

# Run with live output
process = subprocess.Popen(
    CMD,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f'\nTraining exited with code: {process.returncode}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 8: Save & List Output Checkpoints
# ─────────────────────────────────────────────────────────────
import json

print('=== SAVED CHECKPOINTS ===')
for root, dirs, files in os.walk('Model'):
    for f in sorted(files):
        if f.endswith('.pt') or f.endswith('.json'):
            p = os.path.join(root, f)
            print(f'  {p}: {os.path.getsize(p)/(1024*1024):.2f} MB')

# Show final training status
status_path = 'Model/training_status.json'
if os.path.exists(status_path):
    with open(status_path) as fh:
        s = json.load(fh)
    print(f'\n=== FINAL TRAINING STATUS ===')
    print(f'Step    : {s["step"]} / {s["target_steps"]}')
    print(f'Loss    : {s["loss"]:.4f} | EMA Loss: {s["ema_loss"]:.4f}')
    print(f'Speed   : {s["tok_s"]:.1f} tok/s')
    if s.get('validation'):
        v = s['validation']
        print(f'Val Loss: {v["val_loss"]:.4f} | Val Top-1: {v["val_acc"]:.2f}%')

print('\n✅ Training complete! Download checkpoints from the Output tab on the right.')